# codesearch — реранкер: ретривер vs реранкер 512 vs реранкер 2048 (mean ± std)

Корпус **10k** функций на ретривер (Qwen3-0.6B) → топ-**100** → реранкер (Qwen3-Reranker-0.6B).
Три конфигурации на одинаковых данных:
1. **ретривер-only** (без реранкера) — базовая линия;
2. **+реранкер 512** — задушенный (дефолт, на котором гоняла Эльвина, реранкер видит ~400 токенов кода);
3. **+реранкер 2048** — полный (реранкер видит код целиком).

**Каждая конфигурация гоняется N_RUNS раз** на разных случайных выборках запросов (`--seed`) →
получаем **mean ± std**. Eval детерминированный, поэтому просто повторить прогон = то же число;
разные seed дают настоящую статистику по разным выборкам.

**Время:** реранкер медленный. По умолчанию python, 50 запросов × 10 прогонов × 2 реранкер-конфига —
ориентировочно 1–1.5 часа на T4. Уменьши `N_RUNS` или `MAX_QUERIES`, если надо быстрее.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**, затем Runtime → Run all.

In [ ]:
!nvidia-smi

In [ ]:
!rm -rf code-searching-engine
!git clone -b test/reranker-vanya https://github.com/PopovDanil/code-searching-engine.git
%cd code-searching-engine
!pip install -q -U sentence-transformers
!pip install -q rank-bm25 faiss-cpu tree-sitter tree-sitter-python tree-sitter-java tree-sitter-javascript tree-sitter-go tree-sitter-ruby tree-sitter-php datasets typer pyyaml

In [ ]:
import os
import re
import subprocess
from statistics import mean, pstdev

# N_RUNS прогонов на разных случайных выборках (seed) -> mean ± std.
# Eval детерминированный, поэтому без разных seed повторы дали бы одинаковые числа.
LANGUAGES = "python"
MAX_DATASET_RECORDS = "10000"   # корпус на ретривер
MAX_QUERIES = "50"              # запросов на язык в ОДНОМ прогоне
N_RUNS = 10                     # число прогонов (seed 0..N-1)

env = os.environ.copy()
env["PYTHONPATH"] = "src"


def parse_overall(text):
    res, current = {}, None
    tail = text.split("Evaluation Results:")[-1]
    for line in tail.splitlines():
        m_lang = re.match(r"^\s{2}(\S+):\s*$", line)
        m_val = re.match(r"^\s{4}(\S+):\s+([0-9.]+)\s*$", line)
        if m_lang:
            current = m_lang.group(1); res[current] = {}
        elif m_val and current is not None:
            res[current][m_val.group(1)] = float(m_val.group(2))
    return res.get("overall", {})


def run_once(config_path, seed):
    proc = subprocess.run(
        ["python", "-m", "cli", "evaluate",
         "--languages", LANGUAGES,
         "--max-dataset-records", MAX_DATASET_RECORDS,
         "--max-queries", MAX_QUERIES,
         "--seed", str(seed),
         "--config", config_path],
        env=env, capture_output=True, text=True,
    )
    out = proc.stdout + "\n" + proc.stderr
    o = parse_overall(out)
    if proc.returncode != 0 or not o:
        print(f"  seed {seed} FAILED:\n", out[-1500:])
    return o


def run_config(config_path, label):
    print(f"\n{'='*60}\n  {label}  ({config_path}) — {N_RUNS} прогонов\n{'='*60}", flush=True)
    runs = []
    for s in range(N_RUNS):
        o = run_once(config_path, s)
        if o:
            runs.append(o)
            print(f"  seed {s}: R@1={o.get('Recall@1', float('nan')):.4f} "
                  f"MRR={o.get('MRR', float('nan')):.4f} "
                  f"NDCG@10={o.get('NDCG@10', float('nan')):.4f}", flush=True)
    return runs


runs_emb = run_config("configs/vanya-embedding-only.yaml", "РЕТРИВЕР-ONLY")
runs_512 = run_config("configs/vanya-reranker-512.yaml", "+ РЕРАНКЕР задушенный (512)")
runs_2048 = run_config("configs/vanya-reranker.yaml", "+ РЕРАНКЕР полный (2048)")

In [ ]:
# mean ± std по N прогонам: ретривер vs +реранкер(512) vs +реранкер(2048)
METRICS = ["Recall@1", "Recall@5", "Recall@10", "Recall@50", "Recall@100", "MRR", "NDCG@10", "NDCG@100"]


def stat(runs, m):
    vals = [r[m] for r in runs if m in r]
    if not vals:
        return float('nan'), float('nan')
    return mean(vals), (pstdev(vals) if len(vals) > 1 else 0.0)


def cell(runs, m):
    mu, sd = stat(runs, m)
    return f"{mu:.4f} ± {sd:.4f}"


groups = [("ретривер", runs_emb), ("+реранкер 512", runs_512), ("+реранкер 2048", runs_2048)]

print(f"прогонов: emb={len(runs_emb)}, 512={len(runs_512)}, 2048={len(runs_2048)} "
      f"(по {MAX_QUERIES} запросов каждый)\n")
print(f"{'Метрика':<12} " + " ".join(f"{name:>18}" for name, _ in groups))
print("-" * 70)
for m in METRICS:
    print(f"{m:<12} " + " ".join(f"{cell(runs, m):>18}" for _, runs in groups))

print("\nMarkdown для чата (mean ± std):\n")
print("| Метрика | " + " | ".join(name for name, _ in groups) + " |")
print("|---" * (len(groups) + 1) + "|")
for m in METRICS:
    print(f"| {m} | " + " | ".join(cell(runs, m) for _, runs in groups) + " |")

## Как читать

- **R@1 / MRR / NDCG@10** должны заметно вырасти с реранкером — это его работа (поднять правильный
  ответ выше). Если прирост большой — 2048 лечит то, что душило 512.
- **R@100** практически не изменится — реранкер лишь пересортировывает те же 100 кандидатов.
- **R@50/R@10** могут подрасти, если реранкер вытаскивает ответ из глубины пула ближе к верху.